# Can we predict late delivery?
Goal: predict `late_delivery` for a consignment **at dispatch time**.

Key decision up front: `claim_paid_eur` is compensation paid on a claim *settled after* the consignment, so it is a consequence of lateness, not something known at dispatch. It is excluded from the features (we quantify the leak below).

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.shape, df.consignment.is_unique, df.isna().sum().sum())
print(df.dispatched_on.min(), df.dispatched_on.max())
df.describe().T

## Target and leakage check

In [ ]:
print('late rate:', df.late_delivery.mean().round(3))
print(df.groupby(df.claim_paid_eur>0).late_delivery.agg(['mean','size']))
print('AUC of claim_paid_eur alone:', round(roc_auc_score(df.late_delivery, df.claim_paid_eur),3))

A positive claim almost perfectly tracks lateness, which is the signature of leakage (it is only known after the outcome). Dropped.

## Exploration

In [ ]:
df['hour']=df.dispatched_on.dt.hour; df['dow']=df.dispatched_on.dt.dayofweek; df['month']=df.dispatched_on.dt.month
print(df.groupby('carrier').late_delivery.agg(['mean','size']))
fig,ax=plt.subplots(1,4,figsize=(17,3.5))
for a,(c,q) in zip(ax,[('distance_km',8),('weight_kg',8),('hour',None),('dow',None)]):
    x=pd.qcut(df[c],q,duplicates='drop') if q else df[c]
    df.groupby(x,observed=True).late_delivery.mean().plot(marker='o',ax=a); a.set_title(f'late rate by {c}'); a.tick_params(axis='x',rotation=45)
plt.tight_layout()
df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.mean().plot(figsize=(8,3),title='late rate by month',marker='o');

## Setup
Features known at dispatch: carrier, distance, weight, parcels, and calendar features from dispatch time.
Validation is **temporal** (train on earlier, test on the latest 20%), since a deployed model predicts the future. CV on the training part uses forward-chaining splits.

In [ ]:
df['log_dist']=np.log1p(df.distance_km); df['log_wt']=np.log1p(df.weight_kg)
df['wt_per_parcel']=df.weight_kg/df.parcels
num=['distance_km','weight_kg','parcels','log_dist','log_wt','wt_per_parcel','hour','dow','month']; cat=['carrier']
X=df[num+cat]; y=df.late_delivery
cut=int(len(df)*.8); Xtr,Xte,ytr,yte=X[:cut],X[cut:],y[:cut],y[cut:]
print(len(Xtr),len(Xte), ytr.mean().round(3), yte.mean().round(3))
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),cat)])
models={'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic':make_pipeline(pre,LogisticRegression(max_iter=1000)),
 'grad. boosting':make_pipeline(ColumnTransformer([('c',OneHotEncoder(),cat)],remainder='passthrough'),
        HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))}
tscv=TimeSeriesSplit(5); rows={}
for n,m in models.items():
    cv=cross_val_score(m,Xtr,ytr,cv=tscv,scoring='roc_auc')
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]
    rows[n]=dict(cv_auc=cv.mean(),cv_sd=cv.std(),test_auc=roc_auc_score(yte,p),test_pr_auc=average_precision_score(yte,p),test_brier=brier_score_loss(yte,p))
res=pd.DataFrame(rows).T.round(3); res

## Performance and drivers

In [ ]:
p=models['logistic'].predict_proba(Xte)[:,1]
rng=np.random.default_rng(0); yt=yte.values; b=[]
for _ in range(500):
    i=rng.integers(0,len(yt),len(yt)); b.append(roc_auc_score(yt[i],p[i]))
print('logistic test AUC 95%% CI: %.3f-%.3f'%tuple(np.percentile(b,[2.5,97.5])))
fig,ax=plt.subplots(1,2,figsize=(11,4))
for n in ['logistic','grad. boosting']:
    f,t,_=roc_curve(yte,models[n].predict_proba(Xte)[:,1]); ax[0].plot(f,t,label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].legend(); ax[0].set_title('ROC (test)')
pi=permutation_importance(models['grad. boosting'],Xte,yte,scoring='roc_auc',n_repeats=10,random_state=0)
pd.Series(pi.importances_mean,index=X.columns).sort_values().plot.barh(ax=ax[1],title='Permutation importance (AUC drop)')
plt.tight_layout()
d=pd.DataFrame({'p':p,'y':yt}); d['quintile']=pd.qcut(d.p,5,labels=False,duplicates='drop')
d.groupby('quintile').agg(pred=('p','mean'),actual=('y','mean'),n=('y','size')).round(3)

## Why test AUC is so much lower than CV AUC: the base rate drifts
The monthly plot above shows the late rate swinging between ~1% and ~49% and collapsing from Oct 2025 (test set: 2.2% late vs 25.7% in training), while the feature distributions stay flat. Below we check how much signal is in the shipment features versus *when* it was dispatched.

In [ ]:
from sklearn.model_selection import StratifiedKFold
df['t']=(df.dispatched_on-df.dispatched_on.min()).dt.days
feat=['distance_km','weight_kg','parcels','log_dist','log_wt','wt_per_parcel','hour','dow']
def cv_auc(cols, cvs, model=None):
    ct=ColumnTransformer([('n',StandardScaler(),[c for c in cols if c!='carrier']),('c',OneHotEncoder(),['carrier'] if 'carrier' in cols else [])])
    return cross_val_score(make_pipeline(ct,model or LogisticRegression(max_iter=1000)),df[cols],y,cv=cvs,scoring='roc_auc').mean().round(3)
skf=StratifiedKFold(5,shuffle=True,random_state=0)
hgb=lambda: HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)
out={'shipment features only, random CV':cv_auc(feat+['carrier'],skf),
     'shipment features only, forward-chaining CV':cv_auc(feat+['carrier'],tscv),
     'day index only (GBM), random CV':cv_auc(['t'],skf,hgb()),
     'features + day index (GBM), random CV':cv_auc(feat+['carrier','t'],skf,hgb())}
pd.Series(out)

In [ ]:
# is there signal within the test period itself, and within the high-rate period?
for name,mask in [('test period (last 20%)',df.index>=cut),('Mar 2024 - Sep 2025',df.dispatched_on<'2025-10-01')]:
    s=df[mask]; m=make_pipeline(pre,LogisticRegression(max_iter=1000))
    a=cross_val_score(m,s[num+cat],s.late_delivery,cv=StratifiedKFold(5,shuffle=True,random_state=0),scoring='roc_auc').mean()
    print(f'{name}: n={len(s)}, late rate={s.late_delivery.mean():.3f}, within-period CV AUC={a:.3f}')

## Conclusion
**Short answer: only weakly, and not reliably going forward.**

- **Leakage excluded.** `claim_paid_eur` alone gives AUC 0.99 but is settled after the delivery outcome, so it is unusable for prediction at dispatch. Anything built on it would look excellent and be useless.
- **Dispatch-time features carry modest signal.** Carrier (coastal is worst, ~27% late vs ~18-19%), plus distance/weight effects, give a forward-chaining CV AUC of ~0.71-0.73 versus 0.50 for the baseline.
- **That does not hold on the most recent data.** On the temporal hold-out (last 20%, Oct 2025 - Feb 2026) AUC drops to ~0.61 (bootstrap CI roughly 0.53-0.70) for both models, and the predicted risks are badly over-estimated (predicted 6-52% vs. actual 1-3%).
- **Cause: the late rate is driven by something not in the data.** It moves between ~1% and ~49% month to month and collapses after Sep 2025 for every carrier, with no matching shift in distance, weight or parcels. Candidate explanations to check with the data owner: a process or carrier-contract change, seasonal capacity effects, or recent labels not being fully settled yet (a right-censoring issue; the claim rate also falls in the same months).

- **The shipment signal itself is real and stable.** Features-only AUC is ~0.72-0.73 under random CV, forward-chaining CV, and within the test period alone (0.725) or the earlier period (0.745). The hold-out figure of ~0.61 is low mainly because the model was trained at a ~26% base rate and the test set has only ~35 late consignments (wide CI), so it is noisy rather than proof the signal vanished. Dispatch date alone is as predictive (AUC 0.71) as all shipment features, and combining them reaches ~0.81 under random CV, which is not achievable for future dates because the time effect can't be extrapolated.

**Recommendation:** don't deploy this as is. First confirm whether the recent drop is real or a labelling/censoring artifact, and obtain drivers of the time-varying rate (weather, volumes, depot load, promised-date lead time, destination). Then re-evaluate on a time-ordered hold-out. Until then the model is best used only to rank relative risk within a period, not to estimate absolute probabilities.

*Note: the notebook was written and checked cell by cell outside a Jupyter kernel (the sandbox blocked kernel startup); re-run it top to bottom to populate the saved outputs.*